# BER full pipeline — Kaggle GPU (T4/P100)

**Setup**
1. Settings → Accelerator → **GPU** (verify phone on account if greyed out)
2. Add input dataset: competition `dataset/` with `train/` and `test/` (title suggestion: `amazon-ml-ber-dataset`)
3. Run All. Session limit ~9–12h; re-run resumes from `/kaggle/working/work_full` caches.

Winning stack: dense ∪ BM25@5 ∪ script@50 ∪ empty-BM25@40 ∪ housenum ∪ fuzz@24.

In [ ]:
import os, sys, subprocess, shutil
print("GPU check:")
subprocess.run(["nvidia-smi"], check=False)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Locate competition dataset under /kaggle/input
DS = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train" in dirs and "test" in dirs:
        if os.path.isfile(os.path.join(root, "train", "train_source1.tsv")):
            DS = root
            break
assert DS, "Attach a Kaggle Dataset that contains train/ and test/ TSV folders"
print("DS=", DS)
WORK = "/kaggle/working/work_full"
OUT = "/kaggle/working/output"
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Fresh code from GitHub (includes Kaggle-safe runner)
REPO = "https://github.com/Karthikatstuffmama/amazon-ml-challenge-2026.git"
CODE = "/kaggle/working/amazon-ml-challenge-2026"
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
BER = os.path.join(CODE, "code", "business_entity_resolution")
os.chdir(BER)
print("cwd", os.getcwd())

In [ ]:
# Deps (Kaggle images already have torch+cuda often; pin the rest)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentence-transformers==3.3.1", "transformers==4.46.3",
                "faiss-cpu==1.9.0.post1", "anyascii==0.3.2", "jellyfish==1.2.1",
                "rapidfuzz==3.10.1", "lightgbm==4.5.0"], check=True)
print("deps ok")

In [ ]:
# Full pipeline — T4/P100 safe knobs (not A100 settings)
env = os.environ.copy()
env.update({
    "DS": DS,
    "WORK": WORK,
    "OUT": OUT,
    "PY": sys.executable,
    "DENSE_BATCH": "512",
    "WORKERS": "4",
    "OMP_NUM_THREADS": "4",
    "MKL_NUM_THREADS": "4",
    "OPENBLAS_NUM_THREADS": "4",
    "KMP_DUPLICATE_LIB_OK": "TRUE",
    "CUDA_MODULE_LOADING": "LAZY",
    "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:128",
})
cmd = ["bash", "scripts/run_kaggle.sh"]
print("running", " ".join(cmd))
rc = subprocess.call(cmd, env=env)
print("exit", rc)
assert rc == 0, rc

In [ ]:
# Show outputs + pack for download
import pathlib
out = pathlib.Path(OUT)
for p in sorted(out.glob("*")):
    print(p, p.stat().st_size)
rep = pathlib.Path(WORK) / "models" / "report.json"
if rep.exists():
    print(rep.read_text()[:2000])
subprocess.run(["ls", "-lh", OUT], check=False)
# FileLink-friendly zip
shutil.make_archive("/kaggle/working/ber_submission", "zip", OUT)
print("wrote /kaggle/working/ber_submission.zip")